# Autocall payoff logic — demo

This notebook demonstrates the `AutocallNote` pricing engine defined in
`autocall_payoff.py`. The path-generation machinery lives in `autocall.ipynb`;
here we regenerate a compact path set the same way and then drive the payoff
engine.

**Pipeline:** simulate correlated paths → attach a date to each path column →
define product `AutocallTerms` → `AutocallNote(...).price()` → inspect the
per-path audit trail.

## 1. Regenerate underlying paths

Identical setup to `autocall.ipynb`: a 2-asset correlated Black–Scholes–Merton
process simulated on a Sobol grid.

In [1]:
import os, sys
import numpy as np
import QuantLib as ql

# Make the module importable from the project root
sys.path.insert(0, os.path.abspath(".."))
from autocall_payoff import AutocallNote, AutocallTerms, BasketMode, RedemptionType

eval_date = ql.Date(19, 5, 2026)
calendar = ql.UnitedStates(ql.UnitedStates.NYSE)
day_count = ql.Actual365Fixed()
ql.Settings.instance().evaluationDate = eval_date

fwd_rate = ql.SimpleQuote(0.03)
zero_curve = ql.FlatForward(eval_date, ql.QuoteHandle(fwd_rate), day_count, ql.Continuous)
zero_curve_handle = ql.YieldTermStructureHandle(zero_curve)

spot_price_1, spot_price_2 = 295.7, 195.7
spot_handle_1 = ql.QuoteHandle(ql.SimpleQuote(spot_price_1))
spot_handle_2 = ql.QuoteHandle(ql.SimpleQuote(spot_price_2))

div_pillars = [ql.Date(21, 5, 2026), ql.Date(19, 6, 2026), ql.Date(18, 9, 2026), ql.Date(17, 12, 2026)]
div_curve_1 = ql.ZeroCurve(div_pillars, [0.0, 0.10358, 0.07464, 0.05098], day_count, calendar)
div_curve_2 = ql.ZeroCurve(div_pillars, [0.0, 0.15358, 0.09464, 0.08098], day_count, calendar)
div_curve_1.enableExtrapolation(); div_curve_2.enableExtrapolation()
div_handle_1 = ql.YieldTermStructureHandle(div_curve_1)
div_handle_2 = ql.YieldTermStructureHandle(div_curve_2)

vol_dates = [ql.Date(19, 8, 2026), ql.Date(19, 11, 2026), ql.Date(19, 2, 2027), ql.Date(19, 5, 2027),
             ql.Date(19, 8, 2027), ql.Date(19, 11, 2027), ql.Date(19, 2, 2028), ql.Date(19, 5, 2028)]
atm_vols_1 = [0.324566, 0.324566, 0.320002, 0.314524, 0.309046, 0.310071, 0.310603, 0.310603]
vol_curve_1 = ql.BlackVarianceCurve(eval_date, vol_dates, atm_vols_1, day_count)
vol_handle = ql.BlackVolTermStructureHandle(vol_curve_1)

bsm_1 = ql.BlackScholesMertonProcess(spot_handle_1, div_handle_1, zero_curve_handle, vol_handle)
bsm_2 = ql.BlackScholesMertonProcess(spot_handle_2, div_handle_2, zero_curve_handle, vol_handle)

rho = 0.65
corr = ql.Matrix(2, 2)
corr[0][0] = corr[1][1] = 1.0
corr[0][1] = corr[1][0] = rho
multi_process = ql.StochasticProcessArray([bsm_1, bsm_2], corr)

In [2]:
sim_dates = [ql.Date(19, 6, 2026), ql.Date(18, 9, 2026), ql.Date(17, 12, 2026),
             ql.Date(19, 2, 2027), ql.Date(19, 5, 2027), ql.Date(19, 8, 2027),
             ql.Date(19, 11, 2027), ql.Date(19, 2, 2028), ql.Date(19, 5, 2028)]
sim_times = [day_count.yearFraction(eval_date, d) for d in sim_dates]
time_grid = ql.TimeGrid(sim_times, len(sim_times))

n_assets = multi_process.size()
n_steps = len(time_grid) - 1
dimensions = n_assets * n_steps

sobol = ql.UniformLowDiscrepancySequenceGenerator(dimensions, 0, ql.SobolRsg.JoeKuoD7)
gaussian = ql.GaussianLowDiscrepancySequenceGenerator(sobol)
path_gen = ql.GaussianSobolMultiPathGenerator(multi_process, time_grid, gaussian, False)

n_paths = 2 ** 12
paths = np.zeros((n_paths, n_assets, n_steps + 1))
for i in range(n_paths):
    multi_path = path_gen.next().value()
    for a in range(n_assets):
        asset_path = multi_path[a]
        paths[i, a] = [asset_path[t] for t in range(n_steps + 1)]

print("paths.shape:", paths.shape)

# Column 0 is the valuation date; columns 1..n are the simulation dates.
path_dates = [eval_date] + sim_dates
schedule = sim_dates                       # one observation per simulation date
initial_fixings = [spot_price_1, spot_price_2]

paths.shape: (4096, 2, 10)


## 2. Worst-of, 'usual' early redemption

Basket metric = `min` of constituent performances; the note is redeemed when
that worst performer is **at or above** the 100% barrier. Guaranteed coupon of
2% is paid every surviving period; early redemption pays a 4% per-period
snowball coupon.

In [3]:
terms = AutocallTerms(
    autocall_barrier=1.00,
    redemption_type=RedemptionType.USUAL,
    basket_mode=BasketMode.WORST_OF,
    early_redemption_rate=0.04,
    snowball=True,
    guaranteed_coupon=0.02,
    coupon_digital=False,
    coupon_on_redemption=False,
)

note = AutocallNote(
    terms=terms,
    initial_fixings=initial_fixings,
    paths=paths,
    path_dates=path_dates,
    schedule=schedule,
    discount_curve=zero_curve_handle,
)
price = note.price()
note.summary()

{'price': 0.14226895363455483,
 'stderr': 0.0015460349870856512,
 'n_paths': 4096,
 'redemption_prob_total': 0.646484375}

### Per-path audit trail

Every `(path, observation)` row records the constituent performances, basket
metric, redemption flag, coupon, interest, total and discounted cashflow.

In [4]:
# First path that actually autocalls, shown end-to-end
redeemed_paths = note.audit.groupby("path")["redeemed"].any()
example_path = redeemed_paths[redeemed_paths].index[0]
note.audit.loc[example_path].round(4)

,date,perf_0,perf_1,basket_metric,redeemed,coupon,redemption_interest,cashflow,discount_factor,discounted_cf
obs,,,,,,,,,,
0,2026-06-19,1.0208,1.0723,1.0208,True,0.0,0.04,0.04,0.9975,0.0399
1,2026-09-18,0.8641,0.9920,0.8641,False,0.0,0.00,0.00,0.9900,0.0000
2,2026-12-17,0.9865,1.0242,0.9865,False,0.0,0.00,0.00,0.9827,0.0000
3,2027-02-19,0.9427,0.9010,0.9010,False,0.0,0.00,0.00,0.9776,0.0000
4,2027-05-19,1.0723,0.9313,0.9313,False,0.0,0.00,0.00,0.9704,0.0000
5,2027-08-19,1.2173,0.9619,0.9619,False,0.0,0.00,0.00,0.9631,0.0000
6,2027-11-19,1.0526,0.8972,0.8972,False,0.0,0.00,0.00,0.9559,0.0000
7,2028-02-19,1.1046,1.0154,1.0154,False,0.0,0.00,0.00,0.9487,0.0000
8,2028-05-19,1.1584,1.1464,1.1464,False,0.0,0.00,0.00,0.9417,0.0000


In [5]:
# Expected (mean) cashflow profile across all paths
note.expected_cashflows().round(4)

,date,redemption_prob,mean_coupon,mean_interest,mean_cashflow,discount_factor,mean_discounted_cf
obs,,,,,,,
0,2026-06-19,0.3101,0.0138,0.0124,0.0262,0.9975,0.0261
1,2026-09-18,0.1370,0.0111,0.0110,0.0220,0.9900,0.0218
2,2026-12-17,0.0649,0.0098,0.0078,0.0176,0.9827,0.0173
3,2027-02-19,0.0330,0.0091,0.0053,0.0144,0.9776,0.0141
4,2027-05-19,0.0281,0.0085,0.0056,0.0142,0.9704,0.0137
5,2027-08-19,0.0208,0.0081,0.0050,0.0131,0.9631,0.0126
6,2027-11-19,0.0220,0.0077,0.0062,0.0138,0.9559,0.0132
7,2028-02-19,0.0171,0.0073,0.0055,0.0128,0.9487,0.0122
8,2028-05-19,0.0137,0.0071,0.0049,0.0120,0.9417,0.0113


## 3. Memory-on-KO redemption

The note is redeemed once **each** constituent has individually breached the KO
level (= barrier) at least once. For a worst-of basket a breach means the
constituent fixing is at or above the barrier; breaches are memorized for the
remaining life. `basket_metric` here reports how many constituents have breached
so far.

In [6]:
memory_terms = AutocallTerms(
    autocall_barrier=1.00,
    redemption_type=RedemptionType.MEMORY_KO,
    basket_mode=BasketMode.WORST_OF,
    early_redemption_rate=0.04,
    snowball=True,
    guaranteed_coupon=0.02,
)
memory_note = AutocallNote(
    terms=memory_terms,
    initial_fixings=initial_fixings,
    paths=paths,
    path_dates=path_dates,
    schedule=schedule,
    discount_curve=zero_curve_handle,
)
memory_note.price()
memory_note.summary()

{'price': 0.14216758552483244,
 'stderr': 0.0015618253047745422,
 'n_paths': 4096,
 'redemption_prob_total': 0.677490234375}

## 4. Digital guaranteed coupon + multiperformance basket

A weighted-basket ('multiperformance') trigger with a **digital** guaranteed
coupon: the accumulated `c * k` is paid as a single lump on the termination
date instead of period by period.

In [7]:
multi_terms = AutocallTerms(
    autocall_barrier=1.00,
    redemption_type=RedemptionType.USUAL,
    basket_mode=BasketMode.MULTIPERFORMANCE,
    weights=[0.5, 0.5],
    early_redemption_rate=0.04,
    snowball=False,
    guaranteed_coupon=0.02,
    coupon_digital=True,
    coupon_on_redemption=True,
)
multi_note = AutocallNote(
    terms=multi_terms,
    initial_fixings=initial_fixings,
    paths=paths,
    path_dates=path_dates,
    schedule=schedule,
    discount_curve=zero_curve_handle,
)
multi_note.price()
multi_note.expected_cashflows().round(4)

,date,redemption_prob,mean_coupon,mean_interest,mean_cashflow,discount_factor,mean_discounted_cf
obs,,,,,,,
0,2026-06-19,0.4441,0.0089,0.0178,0.0266,0.9975,0.0266
1,2026-09-18,0.1521,0.0061,0.0061,0.0122,0.9900,0.0120
2,2026-12-17,0.0667,0.0040,0.0027,0.0067,0.9827,0.0065
3,2027-02-19,0.0330,0.0026,0.0013,0.0040,0.9776,0.0039
4,2027-05-19,0.0271,0.0027,0.0011,0.0038,0.9704,0.0037
5,2027-08-19,0.0222,0.0027,0.0009,0.0036,0.9631,0.0034
6,2027-11-19,0.0154,0.0022,0.0006,0.0028,0.9559,0.0026
7,2028-02-19,0.0156,0.0025,0.0006,0.0031,0.9487,0.0030
8,2028-05-19,0.0098,0.0403,0.0004,0.0407,0.9417,0.0383
